In [17]:
import numpy as np
import torch
import gymnasium as gym
from dataclasses import dataclass


class CustomMountainCar(gym.Wrapper): # custom wrapper to not use the internal reward
    def __init__(self, env, goal_reward=1.0, step_reward=0.0):
        super().__init__(env)
        self.goal_reward = goal_reward
        self.step_reward = step_reward

    def step(self, action):
        obs, _, terminated, truncated, info = self.env.step(action)
        position, velocity = obs
        goal_position = self.unwrapped.goal_position
        reward = self.goal_reward if position >= goal_position else self.step_reward
        return obs, reward, terminated, truncated, info


@dataclass # typed container for batches of replay data
class ReplayBatch:
    obs: torch.Tensor
    actions: torch.Tensor
    rewards: torch.Tensor
    next_obs: torch.Tensor
    terminated: torch.Tensor
    truncated: torch.Tensor
    episode_id: torch.Tensor
    timestep: torch.Tensor
    indices: torch.Tensor

    def _len_(self):
        return self.obs.shape[0]


class TrajectoryReplayBuffer:
    def __init__(self, capacity, obs_dim, action_dim, device="cpu"):
        self.capacity = capacity
        self.obs_dim = obs_dim
        self.action_dim = action_dim
        self.device = device

        self.obs = np.zeros((capacity, obs_dim), dtype=np.float32)
        self.actions = np.zeros((capacity, action_dim), dtype=np.float32)
        self.rewards = np.zeros((capacity, 1), dtype=np.float32)
        self.next_obs = np.zeros((capacity, obs_dim), dtype=np.float32)
        self.terminated = np.zeros((capacity, 1), dtype=np.float32)
        self.truncated = np.zeros((capacity, 1), dtype=np.float32)

        self.episode_id = np.full((capacity,), -1, dtype=np.int64)
        self.timestep = np.full((capacity,), -1, dtype=np.int64)

        self.pos = 0
        self.size = 0
        self.full = False

        self.current_episode_id = 0
        self.episode_to_indices = {}

    def __len__(self):
        return self.size

    def add_episode(self, episode):
        ep_id = self.current_episode_id
        self.current_episode_id += 1

        ep_indices = []

        T = len(episode["obs"])
        for t in range(T):
            idx = self.pos

            if self.full:
                old_ep = self.episode_id[idx]
                if old_ep in self.episode_to_indices:
                    try:
                        self.episode_to_indices[old_ep].remove(idx)
                        if len(self.episode_to_indices[old_ep]) == 0:
                            del self.episode_to_indices[old_ep]
                    except ValueError:
                        pass

            self.obs[idx] = np.asarray(episode["obs"][t], dtype=np.float32)
            self.actions[idx] = np.asarray(episode["actions"][t], dtype=np.float32).reshape(-1)
            self.rewards[idx] = np.asarray([episode["rewards"][t]], dtype=np.float32)
            self.next_obs[idx] = np.asarray(episode["next_obs"][t], dtype=np.float32)
            self.terminated[idx] = np.asarray([episode["terminated"][t]], dtype=np.float32)
            self.truncated[idx] = np.asarray([episode["truncated"][t]], dtype=np.float32)

            self.episode_id[idx] = ep_id
            self.timestep[idx] = t

            ep_indices.append(idx)

            self.pos = (self.pos + 1) % self.capacity
            if self.size < self.capacity:
                self.size += 1
            else:
                self.full = True

        self.episode_to_indices[ep_id] = ep_indices

    def sample(self, batch_size):
        assert self.size > 0, "Buffer is empty"
        idxs = np.random.randint(0, self.size, size=batch_size)

        return ReplayBatch(
            obs=torch.tensor(self.obs[idxs], device=self.device),
            actions=torch.tensor(self.actions[idxs], device=self.device),
            rewards=torch.tensor(self.rewards[idxs], device=self.device),
            next_obs=torch.tensor(self.next_obs[idxs], device=self.device),
            terminated=torch.tensor(self.terminated[idxs], device=self.device),
            truncated=torch.tensor(self.truncated[idxs], device=self.device),
            episode_id=torch.tensor(self.episode_id[idxs], device=self.device),
            timestep=torch.tensor(self.timestep[idxs], device=self.device),
            indices=torch.tensor(idxs, device=self.device),
        )

    def sample_future_goal_batch(self, batch_size, min_k=1, max_k=None):
        assert self.size > 0, "Buffer is empty"

        valid_indices = []
        future_goal_indices = []

        tries = 0
        max_tries = batch_size * 20

        while len(valid_indices) < batch_size and tries < max_tries:
            idx = np.random.randint(0, self.size)
            ep_id = self.episode_id[idx]
            t = self.timestep[idx]

            if ep_id == -1 or ep_id not in self.episode_to_indices:
                tries += 1
                continue

            ep_idxs = self.episode_to_indices[ep_id]
            ep_len = len(ep_idxs)

            if t >= ep_len - 1:
                tries += 1
                continue

            max_valid_k = ep_len - 1 - t
            if max_k is not None:
                max_valid_k = min(max_valid_k, max_k)

            if max_valid_k < min_k:
                tries += 1
                continue

            k = np.random.randint(min_k, max_valid_k + 1)
            future_t = t + k
            future_idx = ep_idxs[future_t]

            valid_indices.append(idx)
            future_goal_indices.append(future_idx)
            tries += 1

        assert len(valid_indices) > 0, "Could not sample valid future-goal pairs"

        idxs = np.array(valid_indices, dtype=np.int64)
        g_idxs = np.array(future_goal_indices, dtype=np.int64)

        batch = {
            "obs": torch.tensor(self.obs[idxs], device=self.device),
            "actions": torch.tensor(self.actions[idxs], device=self.device),
            "next_obs": torch.tensor(self.next_obs[idxs], device=self.device),
            "goals": torch.tensor(self.obs[g_idxs], device=self.device),
            "rewards": torch.tensor(self.rewards[idxs], device=self.device),
            "terminated": torch.tensor(self.terminated[idxs], device=self.device),
            "truncated": torch.tensor(self.truncated[idxs], device=self.device),
            "episode_id": torch.tensor(self.episode_id[idxs], device=self.device),
            "timestep": torch.tensor(self.timestep[idxs], device=self.device),
            "future_timestep": torch.tensor(self.timestep[g_idxs], device=self.device),
            "indices": torch.tensor(idxs, device=self.device),
            "goal_indices": torch.tensor(g_idxs, device=self.device),
        }
        return batch

    def sample_negative_goals(self, batch_size):
        idxs = np.random.randint(0, self.size, size=batch_size)
        return torch.tensor(self.obs[idxs], device=self.device)

    def stats(self):
        return {
            "size": self.size,
            "capacity": self.capacity,
            "num_episodes": len(self.episode_to_indices),
            "current_episode_id": self.current_episode_id,
        }

In [ ]:
EPISODES = 100
MAX_HORIZON = 200
BUFFER_CAPACITY = 500000
DEVICE = "mps" if torch.backends.mps.is_available() else "cpu"

env = gym.make(
    "MountainCarContinuous-v0",
    max_episode_steps=MAX_HORIZON,
)
env = CustomMountainCar(env)

obs_dim = env.observation_space.shape[0]
action_dim = env.action_space.shape[0]

replay_buffer = TrajectoryReplayBuffer(
    capacity=BUFFER_CAPACITY,
    obs_dim=obs_dim,
    action_dim=action_dim,
    device=DEVICE,
)

episodes = []

for epi in range(EPISODES):
    obs, _ = env.reset()
    done = False
    t = 0

    ep = {k: [] for k in ["obs", "actions", "rewards", "next_obs", "terminated", "truncated"]}

    while not done:
        action = env.action_space.sample().astype(np.float32)
        next_obs, reward, term, trunc, info = env.step(action)

        ep["obs"].append(obs.astype(np.float32))
        ep["actions"].append(action.astype(np.float32))
        ep["rewards"].append(np.float32(reward))
        ep["next_obs"].append(next_obs.astype(np.float32))
        ep["terminated"].append(np.float32(term))
        ep["truncated"].append(np.float32(trunc))

        obs = next_obs
        done = term or trunc
        t += 1

    ep_np = {k: np.asarray(v, dtype=np.float32) for k, v in ep.items()}
    episodes.append(ep_np)
    replay_buffer.add_episode(ep_np)

env.close()

print("Collected episodes:", len(episodes))
print("Replay stats:", replay_buffer.stats())

Collected episodes: 100
Replay stats: {'size': 20000, 'capacity': 500000, 'num_episodes': 100, 'current_episode_id': 100}


In [20]:
batch = replay_buffer.sample(batch_size=256)
s = batch.obs
a = batch.actions
s_next = batch.next_obs

for s,a,s_next in zip(batch.obs, batch.actions, batch.next_obs):
    print("s:", s.cpu().numpy(), "a:", a.cpu().numpy(), "s_next:", s_next.cpu().numpy())

s: [-0.48555863  0.00210798] a: [0.4410913] s_next: [-0.48307368  0.00248494]
s: [-0.53650707 -0.00375736] a: [-0.9231441] s_next: [-0.54155236 -0.00504528]
s: [-0.47431743 -0.00126964] a: [0.97224593] s_next: [-4.7449696e-01 -1.7954095e-04]
s: [-0.49388018  0.00054184] a: [0.22074826] s_next: [-0.4932298   0.00065037]
s: [-0.6677119  -0.00415478] a: [0.6066817] s_next: [-0.6699092  -0.00219727]
s: [-0.49968004  0.01271169] a: [-0.36744338] s_next: [-0.48769876  0.01198129]
s: [-0.32370377  0.00956179] a: [-0.3036078] s_next: [-0.31600836  0.00769542]
s: [-0.5357507  -0.00054996] a: [0.13692108] s_next: [-5.3600413e-01 -2.5346025e-04]
s: [-0.53825206 -0.00251717] a: [0.14852592] s_next: [-0.54043657 -0.00218452]
s: [-0.5599345  -0.00066205] a: [-0.50870806] s_next: [-0.5610876  -0.00115313]
s: [-0.4784496   0.00521668] a: [0.15585163] s_next: [-0.47333676  0.00511287]
s: [-0.5248071 -0.0099352] a: [-0.8795701] s_next: [-0.5360526  -0.01124549]
s: [-0.58745956 -0.00215803] a: [0.0309882

In [ ]:
# Now here is the NN to train the sampled batch data from the replay buffer, the goal is to learn a dynamics model that predicts s_next from (s,a) pairs.

import torch.nn as nn
import torch.optim as optim 
import torch.nn.functional as F


input = torch.cat([batch.obs, batch.actions], dim=-1)
output = batch.next_obs

class StateActionRepresentationModel(nn.Module): # a simple MLP that takes in (s,a) and predicts s_next

    def __init__(self, obs_dim, action_dim, hidden_dim=256):
        super().__init__()
        self.fc1 = nn.Linear(obs_dim + action_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, hidden_dim)
        self.fc3 = nn.Linear(hidden_dim, obs_dim)

    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        return self.fc3(x)


train size: 200 test size: 56
